In [1]:
import pandas as pd
import numpy as np
import os
from functools import reduce

print("=======================================================")
print(" SCRIPT 3: FEATURE ENGINEERING Y NOWCAST (NOM-172)")
print("=======================================================")

# 1. CARGA DE VARIABLES
variables = ['O3', 'NOX', 'SR', 'TOUT', 'WSR', 'WDR', 'PM10', 'PM2.5']
dfs = [pd.read_parquet(f"data/processed/variables/{var}_clean.parquet") for var in variables if os.path.exists(f"data/processed/variables/{var}_clean.parquet")]

df_ml = reduce(lambda left, right: pd.merge(left, right, on=['Date', 'Estacion'], how='inner'), dfs)
df_ml.sort_values(by=['Estacion', 'Date'], inplace=True)
df_ml.reset_index(drop=True, inplace=True)

# 2. FUNCIÓN NOWCAST EXACTA (NOM-172 / EPA)
def calc_nowcast(arr):
    # arr es un numpy array de 12 elementos. arr[11] es la hora actual, arr[0] es hace 11 horas.
    recent_3 = arr[-3:] # Las 3 horas más recientes
    
    # Regla NOM-172: "se debe tener datos para al menos dos de las tres horas más recientes"
    if np.isnan(recent_3).sum() >= 2:
        return np.nan
    
    valid_arr = arr[~np.isnan(arr)]
    if len(valid_arr) == 0:
        return np.nan
        
    c_min, c_max = np.nanmin(arr), np.nanmax(arr)
    w = (c_min / c_max) if c_max > 0 else 0
    w = max(w, 0.5) # El peso mínimo permitido es 0.5
    
    num, den = 0, 0
    for i in range(12):
        c = arr[11 - i] # Vamos hacia atrás en el tiempo
        if not np.isnan(c):
            num += c * (w ** i)
            den += w ** i
            
    return num / den if den > 0 else np.nan

# 3. APLICACIÓN DE NORMATIVA
print("Calculando O3 a 8h y PM a 12h (NowCast Real)...")
estaciones = df_ml.groupby('Estacion')

df_ml['O3_8h'] = estaciones['O3'].rolling(window=8, min_periods=6).mean().reset_index(level=0, drop=True)

# Aplicamos la función matemática optimizada con NumPy (raw=True)
df_ml['PM2.5_12h'] = estaciones['PM2.5'].rolling(window=12).apply(calc_nowcast, raw=True).reset_index(level=0, drop=True)
df_ml['PM10_12h'] = estaciones['PM10'].rolling(window=12).apply(calc_nowcast, raw=True).reset_index(level=0, drop=True)

# 4. VECTORES DE VIENTO
radianes = df_ml['WDR'] * (np.pi / 180)
df_ml['U_Wind'] = -df_ml['WSR'] * np.sin(radianes)
df_ml['V_Wind'] = -df_ml['WSR'] * np.cos(radianes)
df_ml.drop(columns=['WDR', 'WSR'], inplace=True)

# 5. CREACIÓN DE TIME-LAGS (Base + Específicos para ML)
print("Generando rezagos temporales (Time-Lags)...")
# Lags a corto plazo (Estos sí pueden usar el objeto 'estaciones' original)
for lag in [1, 2, 3, 4]:
    df_ml[f'NOX_lag_{lag}'] = estaciones['NOX'].shift(lag)
    df_ml[f'SR_lag_{lag}'] = estaciones['SR'].shift(lag)
for lag in [2, 4]:
    df_ml[f'TOUT_lag_{lag}'] = estaciones['TOUT'].shift(lag)
for lag in [1, 2]:
    df_ml[f'U_Wind_lag_{lag}'] = estaciones['U_Wind'].shift(lag)
    df_ml[f'V_Wind_lag_{lag}'] = estaciones['V_Wind'].shift(lag)

# ---> CORRECCIÓN APLICADA: Volver a agrupar para las columnas nuevas <---
print("Inyectando Lags específicos de CCF...")
df_ml['PM2.5_lag_7'] = df_ml.groupby('Estacion')['PM2.5_12h'].shift(7)
df_ml['PM10_lag_7'] = df_ml.groupby('Estacion')['PM10_12h'].shift(7)
df_ml['SR_lag_5'] = df_ml.groupby('Estacion')['SR'].shift(5)

# 6. LIMPIEZA Y EXPORTACIÓN
df_ml_final = df_ml.dropna().reset_index(drop=True)
os.makedirs("data/ml_ready", exist_ok=True)
df_ml_final.to_parquet("data/ml_ready/dataset_ozono_predictivo.parquet", index=False)
print(f"✅ ¡Dataset Final Generado! Filas listas para modelar: {df_ml_final.shape[0]:,}")

 SCRIPT 3: FEATURE ENGINEERING Y NOWCAST (NOM-172)
Calculando O3 a 8h y PM a 12h (NowCast Real)...
Generando rezagos temporales (Time-Lags)...
Inyectando Lags específicos de CCF...
✅ ¡Dataset Final Generado! Filas listas para modelar: 413,291
